# MVSep-MDX23 Colab Fork v2.5
Adaptation of MVSep-MDX23 algorithm for Colab, with few tweaks:

https://colab.research.google.com/github/jarredou/MVSEP-MDX23-Colab_v2/blob/v2.4/MVSep-MDX23-Colab.ipynb  
<br>  

Recent changes:  


**v2.5**
* Kim's MelBand-Roformer model added  


**v2.4**
* BS-Roformer models from viperx added
* MDX-InstHQ4 model added as optionnal
* Flac output
* Control input volume gain
* Filter vocals below 50Hz option
* Better chunking algo (no clicks)
* Some code cleaning

</font>
<br>

<details>
    <summary>Full changelog :</summary>
<br>
<font size=2>
<br>

[**v2.3**](https://github.com/jarredou/MVSEP-MDX23-Colab_v2/tree/v2.3)
* HQ3-Instr model replaced by VitLarge23 (thanks to MVSep)
* Improved MDXv2 processing (thanks to Anjok)
* Improved BigShifts algo (v2)
* BigShifts processing added to MDXv3 & VitLarge
* Faster folder batch processing

[**v2.2.2**](https://github.com/jarredou/MVSEP-MDX23-Colab_v2/tree/v2.2)
* Improved MDXv3 chunking code (thanks to HymnStudio)
* D1581 demo model replaced by new InstVocHQ MDXv3 model.
<br>

**v2.2.1**
* Added custom weights feature
* Fixed some bugs
* Fixed input: you can use a file or a folder as input now
<br>

**v2.2**
* Added MDXv3 compatibility
* Added MDXv3 demo model D1581 in vocals stem multiband ensemble.
* Added VOC-FT Fullband SRS instead of UVR-MDX-Instr-HQ3.
* Added 2stems feature : output only vocals/instrum (faster processing)
* Added 16bit output format option
* Added "BigShift trick" for MDX models
* Added separated overlap values for MDX, MDXv3 and Demucs
* Fixed volume compensation fine-tuning for MDX-VOC-FT
<br>

[**v2.1 (by deton24)**](https://github.com/deton24/MVSEP-MDX23-Colab_v2.1)
* Updated with MDX-VOC-FT instead of Kim Vocal 2
<br>

[**v2.0**](https://github.com/jarredou/MVSEP-MDX23-Colab_v2/tree/2.0)
* Updated with new Kim Vocal 2 & UVR-MDX-Instr-HQ3 models
* Folder batch processing
* Fixed high frequency bleed in vocals
* Fixed volume compensation for MDX models
<br>
</font>
</details>
<br>

Credits:
* [ZFTurbo/MVSep](https://github.com/ZFTurbo/MVSEP-MDX23-music-separation-model)
* Models by [Demucs](https://github.com/facebookresearch/demucs), [Anjok](https://github.com/Anjok07/ultimatevocalremovergui), [Kimberley Jensen](https://github.com/KimberleyJensen), [aufr33](https://github.com/aufr33) & viperx
* Adaptation & tweaks by [jarredou](https://github.com/jarredou/MVSEP-MDX23-Colab_v2/)
</font>

In [ ]:
print('Installing... This will take between 1 and 15 minutes...')
%cd /content
!git clone https://github.com/GeradeHouse/MVSEP-MDX23-Colab_v2_drum_sep.git MVSEP-MDX23-Colab_v2 &> /dev/null
%cd /content/MVSEP-MDX23-Colab_v2
print('Installing dependencies...')
# !pip install -r requirements.txt &> /dev/null
!pip install -r requirements.txt
print('Installation done!')



Installing... This will take between 1 and 15 minutes...
/content
/content/MVSEP-MDX23-Colab_v2
Installing dependencies...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 15.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 5.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.5/68.5 kB 7.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.1/87.1 kB 10.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 5.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 12.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━

Installation done!


In [ ]:
## Restart runtime now!!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Separation




In [ ]:
"""
BigShifts : Better quality/speed performance with values between 3 and 11, **BUT** 11 doesn't always give the best results. Think about it like seed, different values will give slightly different results.
Higher values = longer processing.

Overlap InstVoc/VitLarge : No big advantage to use high values when BigShifts is already high. If you use BigShifts=1 (regular processing), you can use higher values like 8 or even 16.
Higher values = longer processing.
 *Same goes with overlap_VOCFT, but with values between 0 and 0.95*

Weights : How much importance the result from the given model will have in final results.
"""

# NOTE: This cell shows how to set the arguments for inference.py, ensuring
#       all possible arguments are included (and can be turned on/off).

# Separation setup
from pathlib import Path
import glob
from tqdm.auto import tqdm

# Navigate to the working directory
%cd /content/MVSEP-MDX23-Colab_v2

# ---------------------------------------------------------------------------
# --------------------- ARGUMENT CONFIGURATION SECTION -----------------------
# ---------------------------------------------------------------------------

# (1) Input/Output Configuration
input = '/content/drive/MyDrive/Demucs_seperate_MDX25_input/Geradehouse - House 2.0'  # Path to a file or a folder
output_folder = '/content/drive/MyDrive/Demucs_separated_MDX25_v2.5_drum_sep'        # Where results go

# (2) Audio Output Format
# Options: 'PCM_16', 'FLOAT', or 'FLAC'
output_format = 'FLOAT'

# (3) Separation Mode
# - 'Vocals/Instrumental' produces only 2 stems: vocals + instrumental
# - '4-STEMS' produces vocals, bass, drums, other
Separation_mode = '4-STEMS'

# (4) Input Gain
# Any integer dB value, e.g. 0, -3, -6
input_gain = 0

# (5) Restore Original Gain
# Toggle whether the final output returns to the original volume
restore_gain_after_separation = True

# (6) Filter Vocals Below 50 Hz
# Toggle high-pass filter on vocals to remove sub-bass
filter_vocals_below_50hz = True

# (7) BigShifts Trick
# Higher values => potentially better quality but slower
# 1 means disabled
BigShifts = 7

# (8) Additional GPU/Resource Args
# Toggle these depending on environment
large_gpu = True        # if True, store all models in VRAM (needs ~11GB free)
single_onnx = False     # if True, use only a single ONNX model for vocals
use_cpu = False         # if True, force CPU usage (slower, but avoids GPU issues)

# (9) Overlap Settings for Various Models
# Overlap for 'light' demucs models (4-STEMS mode):
overlap_demucs = 0.6

# Overlap for VOCFT model (range ~0.0 to 0.95)
overlap_VOCFT = 0.1

# Overlap for InstHQ4 model (range ~0.0 to 0.95)
overlap_InstHQ4 = 0.1

# Overlap for VitLarge model (range ~1 to ~16, but recommended ~1 to 2 if BigShifts>1)
overlap_VitLarge = 1

# Overlap for InstVoc model (MDXv3)
overlap_InstVoc = 2

# Overlap for BSRoformer model
overlap_BSRoformer = 2

# (10) BSRoformer Model Version
# 'ep_317_1297' or 'ep_368_1296'
BSRoformer_model = 'ep_368_1296'

# (11) Model Weights (Influence in final ensemble)
weight_BSRoformer = 9.18
weight_Kim_MelRoformer = 10
weight_InstVoc = 3.39
weight_VitLarge = 1
weight_InstHQ4 = 2
weight_VOCFT = 2

# (12) Toggling Model Usage (Ensemble)
use_BSRoformer = True
use_Kim_MelRoformer = True
use_InstVoc = True
use_VitLarge = True
use_InstHQ4 = False
use_VOCFT = False

# (13) New Drum/Instrumental Parameters
# shifts_drum: Adjust quality vs. speed for custom drum separation
shifts_drum = 4

# instrumental_version: Enable/disable combined instrumental (e.g., bass + drums + other)
instrumental_version = False

# ---------------------------------------------------------------------------
# ----------------------- ARGUMENTS -> PARSED FLAGS -------------------------
# ---------------------------------------------------------------------------

# Large GPU / CPU / Single Onnx flags
large_gpu_ = '--large_gpu' if large_gpu else ''
cpu_ = '--cpu' if use_cpu else ''
single_onnx_ = '--single_onnx' if single_onnx else ''

# If the user picks "Vocals/Instrumental" mode, pass --vocals_only
vocals_only = '--vocals_only' if Separation_mode == 'Vocals/Instrumental' else ''

# If user chooses to filter vocals
filter_vocals = '--filter_vocals' if filter_vocals_below_50hz else ''

# If user toggles restore_gain
restore_gain = '--restore_gain' if restore_gain_after_separation else ''

# If user toggles usage of certain models
use_VOCFT_ = '--use_VOCFT' if use_VOCFT else ''
use_VitLarge_ = '--use_VitLarge' if use_VitLarge else ''
use_InstHQ4_ = '--use_InstHQ4' if use_InstHQ4 else ''
use_BSRoformer_ = '--use_BSRoformer' if use_BSRoformer else ''
use_Kim_MelRoformer_ = '--use_Kim_MelRoformer' if use_Kim_MelRoformer else ''
use_InstVoc_ = '--use_InstVoc' if use_InstVoc else ''

# Convert new drum/instrumental parameters to arguments
shifts_drum_ = f'--shifts_drum {shifts_drum}'
instrumental_version_ = '--instrumental_version' if instrumental_version else ''

# ---------------------------------------------------------------------------
# ---------------------------- LAUNCH SEPARATION ----------------------------
# ---------------------------------------------------------------------------

# Process a single file or a directory of files
if Path(input).is_file():
    file_path = input
    Path(output_folder).mkdir(parents=True, exist_ok=True)

    !python inference.py \
        --input_audio "{file_path}" \
        --BSRoformer_model {BSRoformer_model} \
        --weight_BSRoformer {weight_BSRoformer} \
        --weight_Kim_MelRoformer {weight_Kim_MelRoformer} \
        --weight_InstVoc {weight_InstVoc} \
        --weight_InstHQ4 {weight_InstHQ4} \
        --weight_VOCFT {weight_VOCFT} \
        --weight_VitLarge {weight_VitLarge} \
        --overlap_demucs {overlap_demucs} \
        --overlap_VOCFT {overlap_VOCFT} \
        --overlap_InstHQ4 {overlap_InstHQ4} \
        --overlap_VitLarge {overlap_VitLarge} \
        --overlap_InstVoc {overlap_InstVoc} \
        --overlap_BSRoformer {overlap_BSRoformer} \
        --output_format {output_format} \
        --BigShifts {BigShifts} \
        --output_folder "{output_folder}" \
        --input_gain {input_gain} \
        {filter_vocals} \
        {restore_gain} \
        {vocals_only} \
        {large_gpu_} \
        {cpu_} \
        {single_onnx_} \
        {use_VitLarge_} \
        {use_VOCFT_} \
        {use_InstHQ4_} \
        {use_InstVoc_} \
        {use_BSRoformer_} \
        {use_Kim_MelRoformer_} \
        {shifts_drum_} \
        {instrumental_version_}
else:
    file_paths = sorted(glob.glob(input + "/*"))[:]
    input_audio_args = ' '.join([f'"{path}"' for path in file_paths])
    Path(output_folder).mkdir(parents=True, exist_ok=True)

    !python inference.py \
        --input_audio {input_audio_args} \
        --BSRoformer_model {BSRoformer_model} \
        --weight_BSRoformer {weight_BSRoformer} \
        --weight_Kim_MelRoformer {weight_Kim_MelRoformer} \
        --weight_InstVoc {weight_InstVoc} \
        --weight_InstHQ4 {weight_InstHQ4} \
        --weight_VOCFT {weight_VOCFT} \
        --weight_VitLarge {weight_VitLarge} \
        --overlap_demucs {overlap_demucs} \
        --overlap_VOCFT {overlap_VOCFT} \
        --overlap_InstHQ4 {overlap_InstHQ4} \
        --overlap_VitLarge {overlap_VitLarge} \
        --overlap_InstVoc {overlap_InstVoc} \
        --overlap_BSRoformer {overlap_BSRoformer} \
        --output_format {output_format} \
        --BigShifts {BigShifts} \
        --output_folder "{output_folder}" \
        --input_gain {input_gain} \
        {filter_vocals} \
        {restore_gain} \
        {vocals_only} \
        {large_gpu_} \
        {cpu_} \
        {single_onnx_} \
        {use_VitLarge_} \
        {use_VOCFT_} \
        {use_InstHQ4_} \
        {use_InstVoc_} \
        {use_BSRoformer_} \
        {use_Kim_MelRoformer_} \
        {shifts_drum_} \
        {instrumental_version_}


/content/MVSEP-MDX23-Colab_v2

--- Debug environment from sub-process ---
Checking if libcudnn.so.9 and related files exist in /usr/lib64-nvidia/:
total 483M
drwxr-xr-x 2 root root 4.0K Jan 24 15:04 gbm
lrwxrwxrwx 1 root root   29 Jan 24 15:04 libcudadebugger.so.1 -> libcudadebugger.so.535.104.05
-rwxr-xr-x 1 root root 9.8M Jan 24 15:04 libcudadebugger.so.535.104.05
lrwxrwxrwx 1 root root   12 Jan 24 15:04 libcuda.so -> libcuda.so.1
lrwxrwxrwx 1 root root   21 Jan 24 15:04 libcuda.so.1 -> libcuda.so.535.104.05
-rwxr-xr-x 1 root root  28M Jan 24 15:04 libcuda.so.535.104.05
lrwxrwxrwx 1 root root   27 Jan 24 15:04 libEGL_nvidia.so.0 -> libEGL_nvidia.so.535.104.05
-rwxr-xr-x 1 root root 1.3M Jan 24 15:04 libEGL_nvidia.so.535.104.05
lrwxrwxrwx 1 root root   11 Jan 24 15:04 libEGL.so -> libEGL.so.1
lrwxrwxrwx 1 root root   15 Jan 24 15:04 libEGL.so.1 -> libEGL.so.1.1.0
-rwxr-xr-x 1 root root  79K Jan 24 15:04 libEGL.so.1.1.0
-rwxr-xr-x 1 root root 931K Jan 24 15:04 libGLdispatch.so.0
lrwxrw

In [ ]:
from google.colab import runtime

runtime.unassign()